# Photos -> Huggingface
This notebook will read all of the photos from a folder and upload them to a pre-defined repository in huggingface

You need your huggingface API key if you're doing this on your own account.

In [1]:
!uv pip install datasets huggingface_hub pillow

Using Python 3.12.15 environment at: /workspaces/CanopyMap/.venv
Checked 3 packages in 22ms


## Logging into huggingface

In [2]:
from huggingface_hub import login

login()

### Load filenames

In [3]:
import pandas as pd
from pathlib import Path

IMAGE_PATH = Path("plants")
df = pd.DataFrame({"file_name": sorted(p.name for p in IMAGE_PATH.glob("*.jpg"))})
df.head()

,file_name
0,channel5_2026-05-04T12-35-49.jpg
1,channel5_2026-05-04T12-45-49.jpg
2,channel5_2026-05-04T12-55-49.jpg
3,channel5_2026-05-04T13-05-49.jpg
4,channel5_2026-05-04T13-15-49.jpg


### Pull out channel, date, and time

In [4]:
parts = df["file_name"].str.extract(r"(channel\d+)_(\d{4}-\d{2}-\d{2})T(\d{2})-(\d{2})-(\d{2})")

df["channel"] = parts[0]
df["date"] = parts[1]
df["time"] = parts[2] + ":" + parts[3] + ":" + parts[4]
df["suitable"] = True
df.head()

,file_name,channel,date,time,suitable
0,channel5_2026-05-04T12-35-49.jpg,channel5,2026-05-04,12:35:49,True
1,channel5_2026-05-04T12-45-49.jpg,channel5,2026-05-04,12:45:49,True
2,channel5_2026-05-04T12-55-49.jpg,channel5,2026-05-04,12:55:49,True
3,channel5_2026-05-04T13-05-49.jpg,channel5,2026-05-04,13:05:49,True
4,channel5_2026-05-04T13-15-49.jpg,channel5,2026-05-04,13:15:49,True


### Save the metadata that gets pushed to huggingface

In [5]:
df.to_csv(IMAGE_PATH / "metadata.csv", index=False)

### Delete bad files

In [6]:
from PIL import Image

bad = []
for name in df["file_name"]:
    try:
        with Image.open(f"{IMAGE_PATH}/{name}") as im:
            im.load()
    except OSError:
        bad.append(name)

len(bad), bad[:5]

(56,
 ['channel5_2026-06-19T18-37-54.jpg',
  'channel5_2026-06-19T20-07-54.jpg',
  'channel5_2026-06-19T21-17-54.jpg',
  'channel5_2026-06-20T02-07-54.jpg',
  'channel5_2026-06-20T03-57-54.jpg'])

In [7]:
for name in bad:
    Path(f"{IMAGE_PATH}/{name}").unlink()

df = df[~df["file_name"].isin(bad)]
df.to_csv(f"{IMAGE_PATH}/metadata.csv", index=False)
len(df)

5248

### Checking to see if every photo is good to go

In [8]:
from datasets import load_dataset

ds = load_dataset("imagefolder", data_dir=IMAGE_PATH)
print(ds)
ds["train"][0]

/workspaces/CanopyMap/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 5248 examples [00:00, 16130.56 examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'channel', 'date', 'time', 'suitable'],
        num_rows: 5248
    })
})


{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=2560x1920>,
 'channel': 'channel5',
 'date': '2026-05-04',
 'time': '12:35:49',
 'suitable': True}

### Push to huggingface

In [9]:
ds.push_to_hub("regularpooria/canopymap_plants", private=True)

Creating parquet from Arrow format: 100%|██████████| 5/5 [00:01<00:00,  3.70ba/s]
Processing Files (1 / 1): 100%|██████████|  424MB /  424MB, 8.68MB/s  





Processing Files (1 / 1): 100%|██████████|  424MB /  424MB, 8.43MB/s  
New Data Upload: 100%|██████████|  422MB /  422MB, 8.42MB/s  
Validating: ██████████| 100%            
Creating parquet from Arrow format: 100%|██████████| 5/5 [00:01<00:00,  4.21ba/s]
Processing Files (1 / 1): 100%|██████████|  419MB /  419MB, 10.4MB/s  





Processing Files (1 / 1): 100%|██████████|  419MB /  419MB, 10.2MB/s  
New Data Upload: 100%|██████████|  419MB /  419MB, 10.2MB/s  
Validating: ██████████| 100%            
Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00,  6.25ba/s]
Processing Files (1 / 1): 100%|██████████|  301MB /  301MB, 9.28MB/s  




Processing Files (1 / 1): 100%|██████████|  301MB /  301MB, 9.12MB/s  
New Data Upload: 100%|██████████|  301MB /  301MB, 9.12MB/s  
Validating: ██████████| 100%            
Creat

CommitInfo(commit_url='https://huggingface.co/datasets/regularpooria/canopymap_plants/commit/ac0db4d98ad3a201aecea6fad44850ebd9bbf0f2', commit_message='Upload dataset', commit_description='', oid='ac0db4d98ad3a201aecea6fad44850ebd9bbf0f2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/regularpooria/canopymap_plants', endpoint='https://huggingface.co', repo_type='dataset', repo_id='regularpooria/canopymap_plants'), pr_revision=None, pr_num=None)